# Knowledge Loom to FORRT Nanopublications

Converts a [TIB Knowledge Loom](https://knowledgeloom.tib.eu) record into FORRT nanopublications.

**Publishing order** (each step needs URIs from the previous):
1. **FORRT Claims** — one per Knowledge Loom statement
2. **FORRT-KL Replication Study** — links to all Claims via `targetsClaim`
3. **FORRT-KL Replication Outcomes** — one per statement, links to Study via `isOutcomeOf`

## 1. Setup

In [25]:
import json
import re
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

from rdflib import Dataset, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from nanopub import load_profile

In [26]:
# Configuration
LOOM_RESOURCE_ID = "a03gjbxh5v"  # From knowledgeloom.tib.eu/resource/<ID>
LOOM_RESOURCE_ID = "vjea9aobg7"

PROFILE_PATH = "/Users/annef/Documents/ScienceLive/annefou-profile/profile.yml"
NP_CMD = "/Users/annef/Documents/ScienceLive/bin/np"
NP_SERVER = "https://np.knowledgepixels.com/"
OUTPUT_DIR = "../outputs/"

profile = load_profile(PROFILE_PATH)
AUTHOR_URI = URIRef(profile.orcid_id)  # CRITICAL: URIRef not ORCID[...]
AUTHOR_NAME = profile.name
print(f"Profile: {AUTHOR_NAME} ({AUTHOR_URI})")
assert 'orcid.org/orcid.org' not in str(AUTHOR_URI), "Double ORCID!"

Profile: Anne Fouilloux (https://orcid.org/0000-0002-1784-2920)


In [27]:
# Namespaces
PUBLISHER = "https://sciencelive4all.org/"
TEMP_NP = Namespace("https://w3id.org/sciencelive/np")
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
SCHEMA = Namespace("http://schema.org/")
SCIENCELIVE = Namespace("https://w3id.org/sciencelive/o/terms/")

FORRT_CLAIM_TEMPLATE = URIRef("https://w3id.org/np/RAu5uTahAxc0OLBB3vaGwK3OQDDZV7QuWtDlBk0Ea3bco")
FORRT_KL_STUDY_TEMPLATE = URIRef("https://w3id.org/np/RALIq4JelUP-q9BuWONcKMJ87B5n59ppcwhQjl-1dheO4")
FORRT_KL_OUTCOME_TEMPLATE = URIRef("https://w3id.org/np/RAw3XdUhxQJfKBaU-cQhV6c7au4rLd5CSUdbMKTS_FB8g")
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RACJ58Gvyn91LqCKIO9zu1eijDQIeEff28iyDrJgjSJF8")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")

KL_API = "https://knowledgeloom.tib.eu/api/v1"
DTREG_TYPE_MAP = {
    "feeb33ad3e4440682a4d": SCIENCELIVE["dtreg-DataAnalysis"],
    "37182ecfb4474942e255": SCIENCELIVE["dtreg-DataPreprocessing"],
    "5b66cb584b974b186f37": SCIENCELIVE["dtreg-DescriptiveStatistics"],
    "b9335ce2c99ed87735a6": SCIENCELIVE["dtreg-GroupComparison"],
    "286991b26f02d58ee490": SCIENCELIVE["dtreg-RegressionAnalysis"],
    "3f64a93eef69d721518f": SCIENCELIVE["dtreg-CorrelationAnalysis"],
    "c6b413ba96ba477b5dca": SCIENCELIVE["dtreg-MultilevelAnalysis"],
    "6e3e29ce3ba5a0b9abfe": SCIENCELIVE["dtreg-ClassPrediction"],
    "c6e19df3b52ab8d855a9": SCIENCELIVE["dtreg-ClassDiscovery"],
    "5e782e67e70d0b2a022a": SCIENCELIVE["dtreg-AlgorithmEvaluation"],
    "437807f8d1a81b5138a3": SCIENCELIVE["dtreg-FactorAnalysis"]
}
print("Namespaces configured.")

Namespaces configured.


In [28]:
# Helpers
def fetch_json(url):
    with urllib.request.urlopen(urllib.request.Request(url)) as resp:
        return json.loads(resp.read().decode())

def slugify(s):
    return re.sub(r'[^a-zA-Z0-9_-]', '-', s.lower()).strip('-')[:60]

def bind_all(ds):
    for p, n in [("this", TEMP_NP), ("sub", TEMP_NP), ("np", NP), ("dct", DCT),
                  ("nt", NT), ("npx", NPX), ("xsd", XSD), ("rdfs", RDFS),
                  ("prov", PROV), ("foaf", FOAF), ("schema", SCHEMA),
                  ("sciencelive", SCIENCELIVE)]:
        ds.bind(p, n)

def make_head(ds):
    this_np = URIRef(TEMP_NP)
    h = ds.graph(URIRef(TEMP_NP + "/Head"))
    h.add((this_np, RDF.type, NP.Nanopublication))
    h.add((this_np, NP.hasAssertion, URIRef(TEMP_NP + "/assertion")))
    h.add((this_np, NP.hasProvenance, URIRef(TEMP_NP + "/provenance")))
    h.add((this_np, NP.hasPublicationInfo, URIRef(TEMP_NP + "/pubinfo")))
    return this_np

def make_provenance(ds):
    p = ds.graph(URIRef(TEMP_NP + "/provenance"))
    p.add((URIRef(TEMP_NP + "/assertion"), PROV.wasAttributedTo, AUTHOR_URI))

def make_pubinfo(ds, this_np, label, template_uri, introduced_uri=None):
    pi = ds.graph(URIRef(TEMP_NP + "/pubinfo"))
    pi.add((AUTHOR_URI, FOAF.name, Literal(AUTHOR_NAME)))
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    pi.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pi.add((this_np, DCT.creator, AUTHOR_URI))
    pi.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pi.add((this_np, NPX.wasCreatedAt, URIRef(PUBLISHER)))
    pi.add((this_np, RDFS.label, Literal(label)))
    pi.add((this_np, NT.wasCreatedFromTemplate, template_uri))
    pi.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pi.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pi.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    if introduced_uri:
        pi.add((this_np, NPX.introduces, introduced_uri))

def sign_and_publish(trig_file, resource_suffix=None):
    """Sign and publish. Returns (nanopub_uri, resource_uri)."""
    from nanopub import Nanopub, NanopubConf
    conf = NanopubConf(profile=profile)
    np_obj = Nanopub(rdf=trig_file, conf=conf)
    np_obj.sign()
    
    # Store signed version for inspection
    signed_path = trig_file.with_suffix(".signed.trig")
    np_obj.store(signed_path)
    
    np_obj.publish()
    np_uri = np_obj.source_uri
    
    # Build the resource URI from the signed file
    resource_uri = None
    if resource_suffix:
        # The sub: namespace in signed file is np_uri + "/"
        # So resource is np_uri + "/" + suffix
        # But np_uri from library may be short form, get full from signed file
        content = signed_path.read_text()
        import re
        match = re.search(r"prefix sub\d*: <([^>]+)>", content)
        if match:
            resource_uri = match.group(1) + resource_suffix
    
    return np_uri, resource_uri

def validate_trig(trig_file):
    content = trig_file.read_text()
    assert 'orcid.org/https' not in content, f"Double ORCID in {trig_file.name}!"
    assert '10.82209/' not in content, f"Unresolvable TIB DOI in {trig_file.name}!"
    return True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Helpers defined.")

Helpers defined.


## 2. Fetch Knowledge Loom data

In [29]:
kl_data = fetch_json(f"{KL_API}/articles/get_article_by_id/?id={LOOM_RESOURCE_ID}")
article = kl_data.get("article", kl_data)
statements = kl_data.get("statements", [])
datasets = kl_data.get("basises", [])
kl_url = f"https://knowledgeloom.tib.eu/resource/{LOOM_RESOURCE_ID}"

# Source DOI: use dataset/basis DOI (= the source publication)
source_doi = None
for d in datasets:
    if d.get("id", "").startswith("http"):
        source_doi = d["id"]
        break

print(f"Title: {article['name']}")
print(f"Source DOI: {source_doi}")
print(f"KL URL: {kl_url}")
print(f"\nStatements ({len(statements)}):")
for s in statements:
    print(f"  - {s['label'][:80]}")
    print(f"    Type: {s.get('type', {}).get('name', '?')}")

Title: Determinants of Bactrocera oleae abundance in olive groves
Source DOI: https://doi.org/10.1007/s10340-022-01489-1
KL URL: https://knowledgeloom.tib.eu/resource/vjea9aobg7

Statements (2):
  - Based on model results, there was a significant negative relationship between th
    Type: Regression analysis
  - Based on model results, a higher percentage of olive groves in the landscape was
    Type: Regression analysis


## 3. Generate and publish FORRT Claims

One claim per Knowledge Loom statement. **Review the preview before publishing.**

In [30]:
def create_claim(stmt):
    ds = Dataset()
    bind_all(ds)
    this_np = make_head(ds)
    claim_id = slugify(f"kl-claim-{stmt['statement_id']}")
    claim_uri = URIRef(str(TEMP_NP) + "/" + claim_id)
    aida_sentence = stmt["label"]
    aida_uri = URIRef(f"http://purl.org/aida/{urllib.parse.quote(aida_sentence, safe='')}")
    a = ds.graph(URIRef(TEMP_NP + "/assertion"))
    a.add((claim_uri, RDF.type, SCIENCELIVE["FORRT-Claim"]))
    # Auto-map KL analysis type to FORRT claim type
    kl_type = stmt.get("type", {}).get("name", "").lower()
    if any(t in kl_type for t in ["group comparison", "regression", "correlation", "multilevel"]):
        forrt_type = SCIENCELIVE["statistical_significance-FORRT-Claim"]
    elif "algorithm evaluation" in kl_type:
        forrt_type = SCIENCELIVE["model_performance-FORRT-Claim"]
    elif any(t in kl_type for t in ["descriptive", "factor"]):
        forrt_type = SCIENCELIVE["descriptive_pattern-FORRT-Claim"]
    elif "preprocessing" in kl_type:
        forrt_type = SCIENCELIVE["data_quality-FORRT-Claim"]
    else:
        forrt_type = SCIENCELIVE["statistical_significance-FORRT-Claim"]
    a.add((claim_uri, RDF.type, forrt_type))
    a.add((claim_uri, RDFS.label, Literal(aida_sentence)))
    a.add((claim_uri, SCIENCELIVE["asAidaStatement"], aida_uri))
    if source_doi:
        a.add((claim_uri, DCT.source, URIRef(source_doi)))
    make_provenance(ds)
    label = f"FORRT Claim: {aida_sentence[:80]}{'...' if len(aida_sentence) > 80 else ''}"
    make_pubinfo(ds, this_np, label, FORRT_CLAIM_TEMPLATE, claim_uri)
    return ds, label

claim_files = []
for i, stmt in enumerate(statements):
    ds, label = create_claim(stmt)
    f = Path(OUTPUT_DIR) / f"{LOOM_RESOURCE_ID}-claim-{i+1}.trig"
    ds.serialize(destination=str(f), format='trig')
    validate_trig(f)
    claim_files.append(f)
    print(f"  {f.name} — {label}")
print(f"\nGenerated {len(claim_files)} claims.")

  vjea9aobg7-claim-1.trig — FORRT Claim: Based on model results, there was a significant negative relationship between th...
  vjea9aobg7-claim-2.trig — FORRT Claim: Based on model results, a higher percentage of olive groves in the landscape was...

Generated 2 claims.


In [31]:
# Preview first claim
if claim_files:
    print(claim_files[0].read_text())

@prefix dct: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sciencelive: <https://w3id.org/sciencelive/o/terms/> .
@prefix sub: <https://w3id.org/sciencelive/np> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/sciencelive/np/assertion> {
    <https://w3id.org/sciencelive/np/kl-claim-d9h0kie3p9> a sciencelive:FORRT-Claim,
            sciencelive:statistical_significance-FORRT-Claim ;
        rdfs:label "Based on model results, there was a significant negative relationship between the Shannon diversity index and B. oleae abundance." ;
        dct:source <https://doi.org/10.1007/s10340-022-01489-1> ;
        sciencelive:asAidaStatement <http://purl.org/aida/Based%20on%20model%20results%2C%20t

In [32]:
# PUBLISH CLAIMS — run only when satisfied with the preview
claim_uris = []  # Claim RESOURCE URIs (for targetsClaim in Study)
for idx, f in enumerate(claim_files):
    suffix = slugify(f"kl-claim-{statements[idx]['statement_id']}")
    np_uri, resource_uri = sign_and_publish(f, resource_suffix=suffix)
    claim_uris.append(resource_uri)
    print(f"Published: {np_uri}")
    print(f"  Claim resource: {resource_uri}")
print(f"
{len(claim_uris)} claims published.")
print("
Use these claim_uris for targetsClaim in the Study.")

Published: https://w3id.org/np/RA5mDbNSNCyV0vWJJQbscM7lCCILGwqrVsL8XuMM3mbHc
Published: https://w3id.org/np/RApFSeXYamR1ZJ_L5HGZs0asF9Zx4h5K-uqSk7X8gfRCA

2 claims published.


## 4. Generate and publish FORRT-KL Replication Study

Links to all published Claims via `targetsClaim`. **Review before publishing.**

In [ ]:
def create_study():
    ds = Dataset()
    bind_all(ds)
    this_np = make_head(ds)
    study_id = slugify(f"kl-study-{LOOM_RESOURCE_ID}")
    study_uri = URIRef(str(TEMP_NP) + "/" + study_id)
    a = ds.graph(URIRef(TEMP_NP + "/assertion"))
    a.add((study_uri, RDF.type, SCIENCELIVE["FORRT-Replication-Study"]))
    a.add((study_uri, RDF.type, SCIENCELIVE["Reproduction-Study"]))
    label = f"Replication study: {article.get('name', LOOM_RESOURCE_ID)}"
    a.add((study_uri, RDFS.label, Literal(label)))
    # targetsClaim — link to all published claims
    for claim_uri in claim_uris:
        a.add((study_uri, SCIENCELIVE["targetsClaim"], URIRef(claim_uri)))
    # Scope
    abstract = article.get("abstract", "")
    scope = f"Reproduction of analyses from Knowledge Loom record: {article.get('name', LOOM_RESOURCE_ID)}"
    if abstract:
        scope += f". {abstract}"
    a.add((study_uri, SCIENCELIVE["hasScopeDescription"], Literal(scope)))
    # Methodology
    types = set(s.get("type", {}).get("name", "") for s in statements if s.get("type", {}).get("name"))
    method = f"Analysis types: {', '.join(sorted(types))}." if types else ""
    method += " Machine-readable descriptions generated using dtreg and published in the TIB Knowledge Loom."
    a.add((study_uri, SCIENCELIVE["hasMethodologyDescription"], Literal(method.strip())))
    # KL link
    a.add((study_uri, SCIENCELIVE["hasLoomRecord"], URIRef(kl_url)))
    make_provenance(ds)
    make_pubinfo(ds, this_np, label, FORRT_KL_STUDY_TEMPLATE, study_uri)
    return ds, label

ds, label = create_study()
study_file = Path(OUTPUT_DIR) / f"{LOOM_RESOURCE_ID}-study.trig"
ds.serialize(destination=str(study_file), format='trig')
validate_trig(study_file)
print(f"Generated: {study_file.name} — {label}")
print(f"Links to {len(claim_uris)} claims.")

In [ ]:
# Preview study
print(study_file.read_text())

In [ ]:
# PUBLISH STUDY
study_suffix = slugify(f"kl-study-{LOOM_RESOURCE_ID}")
study_np_uri, study_uri = sign_and_publish(study_file, resource_suffix=study_suffix)
print(f"Published Study NP: {study_np_uri}")
print(f"Study resource: {study_uri}")

## 5. Generate and publish FORRT-KL Replication Outcomes

One per statement, links to Study via `isOutcomeOf`. **Review before publishing.**

In [ ]:
def create_outcome(stmt, index):
    ds = Dataset()
    bind_all(ds)
    this_np = make_head(ds)
    outcome_id = slugify(f"kl-outcome-{LOOM_RESOURCE_ID}-{index}")
    outcome_uri = URIRef(str(TEMP_NP) + "/" + outcome_id)
    stmt_label = stmt["label"]
    type_info = stmt.get("type", {})
    type_name = type_info.get("name", "analysis")
    label = f"Outcome ({type_name}): {stmt_label[:60]}{'...' if len(stmt_label) > 60 else ''}"
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    a = ds.graph(URIRef(TEMP_NP + "/assertion"))
    a.add((outcome_uri, RDF.type, SCIENCELIVE["FORRT-Replication-Outcome"]))
    a.add((outcome_uri, RDFS.label, Literal(label)))
    a.add((outcome_uri, SCIENCELIVE["isOutcomeOf"], URIRef(study_uri)))
    a.add((outcome_uri, SCIENCELIVE["hasOutcomeRepository"], URIRef(kl_url)))
    a.add((outcome_uri, SCHEMA.endDate, Literal(today, datatype=XSD.date)))
    a.add((outcome_uri, SCIENCELIVE["hasValidationStatus"], SCIENCELIVE["Validated"]))
    a.add((outcome_uri, SCIENCELIVE["hasConfidenceLevel"], SCIENCELIVE["HighConfidence"]))
    a.add((outcome_uri, SCIENCELIVE["hasConclusionDescription"],
           Literal(f"Knowledge Loom verified: {stmt_label}")))
    a.add((outcome_uri, SCIENCELIVE["hasEvidenceDescription"],
           Literal(f"Machine-readable analysis proof ({type_name}) published in the TIB Knowledge Loom.")))
    a.add((outcome_uri, SCIENCELIVE["hasMachineReadableProof"], URIRef(kl_url)))
    type_hash = type_info.get("type_id", "")
    if type_hash in DTREG_TYPE_MAP:
        a.add((outcome_uri, SCIENCELIVE["hasAnalysisType"], DTREG_TYPE_MAP[type_hash]))
    make_provenance(ds)
    make_pubinfo(ds, this_np, label, FORRT_KL_OUTCOME_TEMPLATE, outcome_uri)
    return ds, label

outcome_files = []
for i, stmt in enumerate(statements):
    ds, label = create_outcome(stmt, i+1)
    f = Path(OUTPUT_DIR) / f"{LOOM_RESOURCE_ID}-outcome-{i+1}.trig"
    ds.serialize(destination=str(f), format='trig')
    validate_trig(f)
    outcome_files.append(f)
    print(f"  {f.name} — {label}")
print(f"\nGenerated {len(outcome_files)} outcomes. All link to study: {study_uri}")

In [ ]:
# Preview first outcome
if outcome_files:
    print(outcome_files[0].read_text())

In [ ]:
# PUBLISH OUTCOMES
outcome_uris = []
for idx, f in enumerate(outcome_files):
    suffix = slugify(f"kl-outcome-{LOOM_RESOURCE_ID}-{idx+1}")
    np_uri, resource_uri = sign_and_publish(f, resource_suffix=suffix)
    outcome_uris.append(resource_uri)
    print(f"Published: {np_uri}")
    print(f"  Outcome resource: {resource_uri}")
print(f"
{len(outcome_uris)} outcomes published.")

## 6. Summary

In [ ]:
print(f"Knowledge Loom record: {kl_url}")
print(f"Article: {article['name']}")
print(f"\nPublished nanopublications:")
print(f"\n  Claims ({len(claim_uris)}):")
for i, uri in enumerate(claim_uris):
    print(f"    {i+1}. {uri}")
print(f"\n  Study:")
print(f"    {study_uri}")
print(f"\n  Outcomes ({len(outcome_uris)}):")
for i, uri in enumerate(outcome_uris):
    print(f"    {i+1}. {uri}")
print(f"\nTotal: {len(claim_uris) + 1 + len(outcome_uris)} nanopublications")